In [1]:
import pandas as pd
import numpy as np

import joblib
import json

df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 122)


In [2]:
def calculate_simulated_success(

    founder_strength,
    funding_strength,
    market_opportunity,
    growth

):

    score = (

          0.25 * founder_strength

        + 0.25 * funding_strength

        + 0.25 * market_opportunity

        + 0.25 * growth

    )

    return round(score,2)

In [3]:
current_score = calculate_simulated_success(

    founder_strength=35,
    funding_strength=40,
    market_opportunity=45,
    growth=50

)

print(current_score)

42.5


In [4]:
def what_if_simulator(

    current_founder,
    current_funding,
    current_market,
    current_growth,

    new_founder=None,
    new_funding=None,
    new_market=None,
    new_growth=None

):

    baseline = calculate_simulated_success(

        current_founder,
        current_funding,
        current_market,
        current_growth

    )

    founder = (
        new_founder
        if new_founder is not None
        else current_founder
    )

    funding = (
        new_funding
        if new_funding is not None
        else current_funding
    )

    market = (
        new_market
        if new_market is not None
        else current_market
    )

    growth = (
        new_growth
        if new_growth is not None
        else current_growth
    )

    future = calculate_simulated_success(

        founder,
        funding,
        market,
        growth

    )

    improvement = round(
        future - baseline,
        2
    )

    return baseline, future, improvement

In [5]:
baseline, future, gain = what_if_simulator(

    current_founder=35,
    current_funding=40,
    current_market=45,
    current_growth=50,

    new_founder=70

)

print("Current:", baseline)

print("Future:", future)

print("Gain:", gain)

Current: 42.5
Future: 51.25
Gain: 8.75


In [6]:
def simulate_best_case(row):

    baseline = calculate_simulated_success(

        row["founder_strength_score"],
        row["funding_strength_score"],
        row["market_opportunity_score"],
        row["growth_score"]

    )

    future = calculate_simulated_success(

        100,
        100,
        100,
        100

    )

    return baseline, future

In [7]:
results = df.apply(
    simulate_best_case,
    axis=1
)

df["current_simulated_score"] = [

    x[0]
    for x in results
]

df["best_possible_score"] = [

    x[1]
    for x in results
]

df["max_improvement"] = (

    df["best_possible_score"]

    -

    df["current_simulated_score"]

)

In [8]:
df[[
    "startup_name",

    "current_simulated_score",

    "best_possible_score",

    "max_improvement"

]].head(10)

,startup_name,current_simulated_score,best_possible_score,max_improvement
0,HyperAnalytics Technologies,37.38,100.0,62.62
1,HealthConnect Solutions,26.47,100.0,73.53
2,InsightLoop Ventures,38.41,100.0,61.59
3,BioHub Analytics,23.76,100.0,76.24
4,WaveGen Private Limited,32.31,100.0,67.69
5,LaunchEngine Innovations,29.69,100.0,70.31
6,SolarGrowth Solutions,20.19,100.0,79.81
7,PulseSystems Solutions,24.60,100.0,75.40
8,BridgeVertex Solutions,26.63,100.0,73.37
9,BlueEdge Labs,13.56,100.0,86.44


In [9]:
df.to_csv(

    "../datasets/cleaned/startup_info.csv",

    index=False

)

print("Simulator Data Added ✅")

Simulator Data Added ✅


In [11]:
simulator = {

    "engine_name":
        "What If Simulator",

    "version":
        "V1"

}

joblib.dump(

    simulator,

    "../models/what_if_simulator_model/what_if_simulator.pkl"

)

['../models/what_if_simulator_model/what_if_simulator.pkl']

In [12]:
metadata = {

    "engine_name":
        "What If Simulator",

    "inputs":[

        "founder_strength_score",

        "funding_strength_score",

        "market_opportunity_score",

        "growth_score"

    ],

    "outputs":[

        "current_simulated_score",

        "best_possible_score",

        "max_improvement"

    ]

}

with open(

    "../models/what_if_simulator_model/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

In [13]:
df[[
    "current_simulated_score",
    "best_possible_score",
    "max_improvement"
]].describe()

,current_simulated_score,best_possible_score,max_improvement
count,50000.000000,50000.0,50000.000000
mean,42.392182,100.0,57.607818
std,7.758559,0.0,7.758559
min,7.600000,100.0,28.390000
25%,37.120000,100.0,52.360000
50%,42.440000,100.0,57.560000
75%,47.640000,100.0,62.880000
max,71.610000,100.0,92.400000
